# Check the crossing-change witnesses

Run all cells locally from the repository or in Google Colab. In Colab the
notebook downloads the repository if needed. It checks the 15 witnesses in
`results/bounds.csv`, including the 14 target unknotting witnesses shared by
their targets. The expected result is **15 PASS_SUPPORTED** records.

The checker allows mirrors and requires zero crossings with exactly one
component for the target unknot. It uses ordinary SnapPy computations,
without Sage's `verified=True` mode. It does not load a model or change the
search workbook. Results are saved to `outputs/witness_checks.json`.


In [ ]:
%pip -q install snappy==3.3.2


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

def has_checker(path):
    return ((path / "scripts/check_witnesses.py").is_file()
            and (path / "results/bounds.csv").is_file())

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((path for path in candidates if has_checker(path)), None)
if REPO_ROOT is None:
    try:
        from google.colab import files as colab_files
    except ImportError:
        raise FileNotFoundError("Open this notebook from the upperbounds repository.")
    REPO_ROOT = Path("/content/upperbounds")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/dtubbenhauer/upperbounds.git", str(REPO_ROOT)], check=True)
    if not has_checker(REPO_ROOT):
        raise FileNotFoundError("The repository copy does not contain the witness files. Check the repository URL and folder contents.")
print("Repository:", REPO_ROOT)
REPORT_PATH = REPO_ROOT / "outputs/witness_checks.json"


In [ ]:
completed = subprocess.run([
    sys.executable, str(REPO_ROOT / "scripts/check_witnesses.py"),
    "--repo-root", str(REPO_ROOT), "--output", str(REPORT_PATH),
])
report = json.loads(REPORT_PATH.read_text())
print(json.dumps({key: report.get(key) for key in ["status", "counts", "error"]}, indent=2))
if completed.returncode:
    print("Some checks are unresolved or invalid. Inspect and retain the report; do not treat these as passes.")


In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Saved report:", REPORT_PATH)
else:
    files.download(str(REPORT_PATH))
